<a href="https://colab.research.google.com/github/kjmobile/lb2/blob/main/7_Cross_Validation_and_Grid_Search_Late_Flights_simple.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 7. Validation, Cross-Validation and Grid Search

In notebook 6 we chose the tree's depth by looking at the **test set**. That breaks the rule that the test set is
a "final exam" taken once, at the very end. If we use it to make choices, its score is no longer a fair estimate
of how the model will do on future flights.

This notebook makes those choices with the **training data only**, and opens the test set once, at the end.

Slides: W3.1, validation set, k-fold cross-validation, grid search vs. random search.

> **Data.** The same 2,000 U.S. domestic flights from 2024 as in notebook 6 (BTS on-time records, public data).
> `late` = 1 if the flight arrived 15+ minutes late.

> **Figures.** The figures in this notebook are pictures made by its full version (`7_Cross_Validation_and_Grid_Search_Late_Flights_all.ipynb`), so you can focus on the code that fits the model and produces the results.

## 1. Data and the Test Set We Will Not Touch

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

flights = pd.read_csv('https://raw.githubusercontent.com/kjmobile/lb2/main/data/flights_late_2024.csv')
X = flights.drop(columns='late')
y = flights['late']

# the same split as notebook 6: the test set is put away until section 6
train_X, test_X, train_y, test_y = train_test_split(X, y, test_size=0.2, random_state=320, stratify=y)
print('train:', train_X.shape, ' test:', test_X.shape)

train: (1600, 5)  test: (400, 5)


## 2. A Validation Set

Split the **training** flights again: 80% to fit the trees (`sub`), 20% to compare them (`val`, the
**validation set**). So the model is fit on 80% × 80% = 64% of all flights.

In [2]:
sub_X, val_X, sub_y, val_y = train_test_split(train_X, train_y, test_size=0.2, random_state=320, stratify=train_y)
print('sub:', sub_X.shape, ' validation:', val_X.shape)

sub: (1280, 5)  validation: (320, 5)


In [3]:
rows = []
for depth in [1, 2, 3, 4, 5, 6, 8, 10, None]:      # None = no limit
    t = DecisionTreeClassifier(max_depth=depth, random_state=320)
    t.fit(sub_X, sub_y)
    rows.append({'max_depth': str(depth), 'validation accuracy': t.score(val_X, val_y)})

pd.DataFrame(rows).round(3)

,max_depth,validation accuracy
0,1,0.850
1,2,0.850
2,3,0.862
3,4,0.859
4,5,0.862
5,6,0.859
6,8,0.856
7,10,0.841
8,None,0.781


The validation set lets us compare depths without touching the test set. But it is small (320 flights), so the
ranking can change with a different random split. Cross-validation uses the data more fully.

## 3. K-Fold Cross-Validation

Cut the training flights into $k$ equal parts (**folds**). Fit the model $k$ times; each time one fold is held out
for scoring and the other $k-1$ folds are used for fitting. Average the $k$ scores. Every training flight is used
for scoring exactly once.

In [4]:
from sklearn.model_selection import cross_validate

tree3 = DecisionTreeClassifier(max_depth=3, random_state=320)
scores = cross_validate(tree3, train_X, train_y, cv=5)

print('five fold scores:', scores['test_score'].round(3))
print('average:', round(scores['test_score'].mean(), 3))

five fold scores: [0.866 0.847 0.875 0.872 0.856]
average: 0.863


Careful with the name: in `cross_validate`, `test_score` means the score on the **held-out fold**, not on our test
set, which is still put away. For classification, `cv=5` keeps the share of late flights the same in every fold
(stratified folds).

Now compare depths by their cross-validation average:

In [5]:
rows = []
for depth in [1, 2, 3, 4, 5, 6, 8, 10, None]:
    cv = cross_validate(DecisionTreeClassifier(max_depth=depth, random_state=320), train_X, train_y, cv=5)
    rows.append({'max_depth': str(depth), 'cv accuracy': cv['test_score'].mean()})

cv_by_depth = pd.DataFrame(rows).round(3)
cv_by_depth

,max_depth,cv accuracy
0,1,0.861
1,2,0.861
2,3,0.863
3,4,0.858
4,5,0.862
5,6,0.862
6,8,0.844
7,10,0.832
8,None,0.783


**Figure 1**

![Figure 1](https://raw.githubusercontent.com/kjmobile/lb2/main/data/img/7_fig1.png)

## 4. Grid Search: Try Every Combination

A **hyperparameter** is a setting we choose before fitting, such as `max_depth`. `GridSearchCV` tries every
combination in a grid, scores each one by cross-validation, and keeps the best. Then it refits that model on all
the training flights.

In [6]:
from sklearn.model_selection import GridSearchCV

params = {'max_depth': [2, 3, 4, 5, 6, 8, None],
          'min_samples_leaf': [1, 5, 20, 50]}       # smallest number of flights allowed in a final group
grid = GridSearchCV(DecisionTreeClassifier(random_state=320), params, cv=5)
grid.fit(train_X, train_y)

print('combinations tried:', len(grid.cv_results_['params']), ' fits:', len(grid.cv_results_['params']) * 5)
print('best settings:', grid.best_params_)
print('best cv accuracy:', round(grid.best_score_, 3))

combinations tried: 28  fits: 140
best settings: {'max_depth': 4, 'min_samples_leaf': 5}
best cv accuracy: 0.866


## 5. Random Search: Try a Random Sample

With many hyperparameters, a full grid becomes huge. `RandomizedSearchCV` tries a fixed number of random
combinations (`n_iter`) drawn from ranges we give it. It often finds a setting as good as the grid at a fraction
of the cost, and it can land on values the grid did not list.

In [7]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

params_random = {'max_depth': randint(2, 20),            # any whole number from 2 to 19
                 'min_samples_leaf': randint(1, 100)}    # any whole number from 1 to 99
search = RandomizedSearchCV(DecisionTreeClassifier(random_state=320), params_random, n_iter=20, cv=5,
                            random_state=320)
search.fit(train_X, train_y)

print('best settings:', search.best_params_)
print('best cv accuracy:', round(search.best_score_, 3))

best settings: {'max_depth': 5, 'min_samples_leaf': 6}
best cv accuracy: 0.868


Here the random search found a slightly higher cross-validation score than the grid, with settings the grid did not
list (depth 5, at least 6 flights per final group). The difference is small; both searches agree that a shallow
tree with a few flights per group works best.

## 6. The Final Exam: Open the Test Set Once

All choices are made. Now, and only now, score the chosen model on the test set.

In [8]:
best_tree = grid.best_estimator_      # already refit on all training flights
print('test accuracy of the chosen tree:', round(best_tree.score(test_X, test_y), 3))
print('baseline (always guess "on time"):', round(1 - test_y.mean(), 3))

test accuracy of the chosen tree: 0.858
baseline (always guess "on time"): 0.792


The test score is a little below the best cross-validation score. That is normal: the best of many tries is
partly lucky, so its own score runs a bit high. The test set gives the honest number to report.

### Your turn

Does the number of folds matter? Run **10-fold** cross-validation for the depth-3 tree (`tree3`) and compare its
average with the 5-fold average in section 3.

1. In the cell below, fill in the blanks (`___`) and run it.
2. Then finish this sentence:
   *With 10 folds the average accuracy is ___, against ___ with 5 folds.*

<details><summary>Show answer</summary>

```python
scores_10 = cross_validate(tree3, train_X, train_y, cv=10)
print('ten fold scores:', scores_10['test_score'].round(3))
print('average:', round(scores_10['test_score'].mean(), 3))
```

*With 10 folds the average accuracy is **0.862**, against **0.863** with 5 folds. Almost the same: the number of folds hardly changes the average here, but the single-fold scores spread more with smaller folds (from **0.825** to **0.888**).*
</details>

In [ ]:
scores_10 = cross_validate(tree3, train_X, train_y, cv=___)
print('ten fold scores:', scores_10['___'].round(3))
print('average:', round(scores_10['test_score'].mean(), 3))

## 7. Summary

| Tool | What it does | Here |
|---|---|---|
| Test set | Final exam, opened once at the end | 400 flights |
| Validation set | Compares settings without touching the test set | 320 of the 1,600 training flights |
| K-fold cross-validation | Every training flight is held out once; average the k scores | 5 folds |
| Grid search | Tries every combination in a grid, scored by cross-validation | 28 combinations × 5 folds = 140 fits; best: depth 4 |
| Random search | Tries a random sample of combinations | 20 tries; best: depth 5, slightly higher cv score |
| Final test | The chosen tree on the 400 test flights | about 0.86, against 0.79 for always guessing "on time" |

---
# Challenge

1. Add `'criterion': ['gini', 'entropy']` to the grid in section 4. How many fits does the search run now? Does the
   best setting change?
2. Run the random search with `n_iter=100`. Does it beat the grid search? Is it worth five times the work?
3. Why is it wrong to report the best cross-validation score as "how accurate the model will be on future flights"?

In [10]:
# Your answer